# M6 SLA Risk Board — Professional Breach Intelligence

## Purpose
Contract-anchored SLA compliance analysis with assignment age tracking and professional KPI reporting.

## Core Metrics

### 1. SLA Breach Analysis
**Formula**:
```
SLA_Target_Hours = {P1: 4, P2: 8, P3: 72, P4: 120}
Elapsed_% = (mttr_hours / SLA_Target_Hours[priority]) * 100
Breached = made_sla_bool == FALSE
Breach_Count = COUNT(Breached)
Breach_Rate % = (Breach_Count / Total) * 100
```

### 2. Assignment Ownership Age
**Formula** (NEW):
```
Ownership_Age_Hours = (now - last_assignment_date) in hours
Avg_Ownership = AVG(Ownership_Age_Hours)
Max_Ownership = MAX(Ownership_Age_Hours)
Long_Held = COUNT(Ownership_Age_Hours > 168 hours [7 days])
```

**Escalation Triggers**:
- Green: <24h (fresh assignment)
- Amber: 24-72h (normal)
- Red: 72-168h (needs attention)
- Critical: >168h (escalate immediately)

### 3. On-Hold SLA Clock Analysis
**Challenge**: On Hold pauses SLA clock, masking real breach risk
```
Hold_Duration = (hold_end_date - hold_start_date) in hours
Elapsed_On_Hold_% = (Hold_Duration / Total_Age) * 100
Restart_Risk = Incidents with Elapsed_% >= 75% AND state = 'On Hold'
```

**Restart Risk Logic**:
- If On Hold >14 days: HIGH risk (SLA will breach immediately on release)
- If Elapsed >= 75% On Hold: CRITICAL (single status change causes breach)

### 4. Reassignment Impact Analysis
**Formula**:
```
Avg_MTTR_by_Reassignments:
  - 0 reassignments: Baseline MTTR
  - 1 reassignment: +4-6h average
  - 2+ reassignments: +8-12h average

Breach_Rate_by_Reassignments:
  - Each reassignment increases breach risk by ~15%
```

**Root Cause**: Queue delays, skill mismatches, poor routing rules

### 5. Contract KPI Scorecard
**Four Critical KPIs**:

1. **Aging KPI**: 95% of incidents resolved within 30 days
   ```
   Compliance = (COUNT(resolved_in_30d) / COUNT(resolved)) * 100
   Target: >=95%
   ```

2. **First-Time-Right**: <=1% reopen rate
   ```
   Reopen_Rate = (COUNT(reopen_count > 0) / COUNT(resolved)) * 100
   Target: <=1%
   ```

3. **P1/P2 Zero Tolerance**:
   ```
   P1_Breaches + P2_Breaches = 0
   Critical incident (contractual violation)
   ```

4. **Bug-Linked Escalation**:
   ```
   Operationally_Unresolvable = COUNT(bug_linked_tickets)
   Action: Move to Change Management, don't keep in ops queue
   ```

## Tower & SDM Analysis
- **By Tower**: Aggregate all metrics across towers
- **By SDM**: Track individual service delivery manager performance
- **Comparative**: Identify top vs bottom performers
- **Trending**: Monthly scorecard evolution


In [ ]:
import requests
import pandas as pd

# Fetch SLA data
API_BASE = 'http://127.0.0.1:8002/api/breach'

# Get KPIs
response = requests.get(f'{API_BASE}/kpis')
kpis = response.json()

print('SLA KPIs:')
print(f'  Total at-risk: {kpis.get("total", 0)}')
print(f'  Already breached: {kpis.get("already_breached", 0)}')
print(f'  On Hold: {kpis.get("on_hold", 0)} (restart risk: {kpis.get("hold_restart_risk", 0)})')
print(f'  Avg assignment age: {kpis.get("avg_assignment_age_h", 0):.1f}h')

In [ ]:
# Get compliance by priority
response = requests.get(f'{API_BASE}/sla-compliance')
compliance = response.json()

df = pd.DataFrame(compliance)
print('\nSLA Compliance by Priority:')
print(df[['priority', 'sla_target', 'total', 'compliant', 'compliance_pct']].to_string())